Import bibliotek

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

Funkcje

In [2]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(name)

    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred), 4))
    print("F1:", round(f1_score(y_test, y_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))


def get_metrics(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba)
    }

Wczytanie danych

In [3]:
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test = pd.read_csv('../data/processed/X_test_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

Test wczytania danych

In [4]:
display(X_train.shape)
display(X_test.shape)
display(y_train.shape)
display(y_test.shape)

display(X_train.head())
display(y_train.value_counts())
display(y_test.value_counts())

(8000, 8)

(2000, 8)

(8000,)

(2000,)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min]
0,1,0,0.874402,-0.800498,-0.100148,-0.851496,-1.182391,1.485820
1,1,0,-0.408876,1.900381,-0.122496,-0.329786,-0.421646,1.973739
2,0,1,0.536697,-0.500400,-0.167192,0.011332,0.052397,-1.300041
3,0,1,-0.071171,1.100121,-0.692367,1.325640,1.499616,1.359905
4,1,0,0.198993,-0.900531,-0.223061,-0.189326,-0.280724,-0.733425


Machine failure
0    7736
1     264
Name: count, dtype: int64

Machine failure
0    1934
1      66
Name: count, dtype: int64

Tworzenie modelu naiwnego

In [5]:
dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]

In [6]:
evaluate_model('Dummy Classifier', dummy, X_test, y_test)

Dummy Classifier
Accuracy: 0.967
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
PR-AUC: 0.033
Confusion Matrix:
 [[1934    0]
 [  66    0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98      1934
           1       0.00      0.00      0.00        66

    accuracy                           0.97      2000
   macro avg       0.48      0.50      0.49      2000
weighted avg       0.94      0.97      0.95      2000



Wnioski:



Logistic Regression

In [7]:
log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]


In [8]:
evaluate_model('LogisticRegression', log_reg, X_test, y_test)

LogisticRegression
Accuracy: 0.8475
Precision: 0.1595
Recall: 0.8485
F1: 0.2686
ROC-AUC: 0.9202
PR-AUC: 0.4192
Confusion Matrix:
 [[1639  295]
 [  10   56]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.85      0.91      1934
           1       0.16      0.85      0.27        66

    accuracy                           0.85      2000
   macro avg       0.58      0.85      0.59      2000
weighted avg       0.97      0.85      0.89      2000



Wnioski:



Random Forest

In [9]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

random_forest.fit(X_train, y_train)

evaluate_model('RandomForestClassifier', random_forest, X_test, y_test)

RandomForestClassifier
Accuracy: 0.99
Precision: 0.8594
Recall: 0.8333
F1: 0.8462
ROC-AUC: 0.991
PR-AUC: 0.893
Confusion Matrix:
 [[1925    9]
 [  11   55]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      1934
           1       0.86      0.83      0.85        66

    accuracy                           0.99      2000
   macro avg       0.93      0.91      0.92      2000
weighted avg       0.99      0.99      0.99      2000



Gradient Boosting


In [10]:
gradient_boosting = GradientBoostingClassifier(
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

evaluate_model('GradientBoostingClassifier', gradient_boosting, X_test, y_test)

GradientBoostingClassifier
Accuracy: 0.9895
Precision: 0.8814
Recall: 0.7879
F1: 0.832
ROC-AUC: 0.9943
PR-AUC: 0.9145
Confusion Matrix:
 [[1927    7]
 [  14   52]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      1934
           1       0.88      0.79      0.83        66

    accuracy                           0.99      2000
   macro avg       0.94      0.89      0.91      2000
weighted avg       0.99      0.99      0.99      2000



Porównanie modeli

In [11]:
results = []

results.append(get_metrics("Dummy", dummy, X_test, y_test))
results.append(get_metrics("Logistic Regression", log_reg, X_test, y_test))
results.append(get_metrics("Random Forest", random_forest, X_test, y_test))
results.append(get_metrics("Gradient Boosting", gradient_boosting, X_test, y_test))

results_df = pd.DataFrame(results)
display(results_df.round(4))

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Dummy,0.9670,0.0000,0.0000,0.0000,0.5000,0.0330
1,Logistic Regression,0.8475,0.1595,0.8485,0.2686,0.9202,0.4192
2,Random Forest,0.9900,0.8594,0.8333,0.8462,0.9910,0.8930
3,Gradient Boosting,0.9895,0.8814,0.7879,0.8320,0.9943,0.9145


In [12]:
#results_df.to_csv("../data/processed/model_results_baseline.csv", index=False)

## Analiza przeoczonych awarii - False Negatives

In [13]:
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv('../data/processed/produkcja_clean.csv')

X_raw = df_clean.drop(columns=['Machine failure'])
y_raw = df_clean['Machine failure']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

X_test_raw = X_test_raw.copy()

In [14]:
print("X_test scaled shape:", X_test.shape)
print("X_test raw shape:", X_test_raw.shape)

print("y_test shape:", y_test.shape)
print("y_test_raw shape:", y_test_raw.shape)

X_test scaled shape: (2000, 8)
X_test raw shape: (2000, 8)
y_test shape: (2000,)
y_test_raw shape: (2000,)


In [15]:
print((y_test.reset_index(drop=True) == y_test_raw.reset_index(drop=True)).all())

True


In [16]:
debug_df = X_test_raw.copy()
debug_df['y_true'] = y_test_raw.values

models = {
    'log_reg': log_reg,
    'random_forest': random_forest,
    'gradient_boosting': gradient_boosting
}

for name, model in models.items():
    debug_df[f"{name}_pred"] = model.predict(X_test)
    debug_df[f"{name}_proba"] = model.predict_proba(X_test)[:, 1]

display(debug_df.head())
display(debug_df.shape)

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
7566,1,0,311.0,10.7,1613,34.7,5861.28,142,0,0,0.041565,0,0.000,0,0.000622
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0,0.070138,0,0.150,0,0.138585
9316,0,0,309.4,10.9,1417,41.2,6113.58,155,0,0,0.086594,0,0.005,0,0.000668
2882,1,0,309.6,9.0,1684,32.8,5784.22,53,0,0,0.032116,0,0.000,0,0.000638
4293,1,0,310.0,8.2,1393,44.9,6549.77,208,0,1,0.879893,0,0.410,0,0.037122


(2000, 15)

In [17]:
debug_df[[
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba"
]].head(10)

,y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
7566,0,0,0.041565,0,0.000,0,0.000622
8199,1,0,0.070138,0,0.150,0,0.138585
9316,0,0,0.086594,0,0.005,0,0.000668
2882,0,0,0.032116,0,0.000,0,0.000638
4293,0,1,0.879893,0,0.410,0,0.037122
4261,0,0,0.183033,0,0.000,0,0.002261
4485,0,0,0.090754,0,0.000,0,0.002235
2518,0,0,0.115763,0,0.000,0,0.000668
8084,0,0,0.095162,0,0.000,0,0.000622
9661,0,0,0.306636,0,0.085,0,0.054976
